[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/08_docente_hiperparametros.ipynb)

# MLY1101 · Machine Learning — Actividad 3.1
## Ajuste de hiperparámetros

**Resultado de aprendizaje (RA3):** elabora soluciones avanzadas de aprendizaje automático
mediante la optimización de hiperparámetros, técnicas de ensamble y validación cruzada, para
garantizar la precisión y generalización del modelo frente a objetivos de negocio complejos.

**Indicador de logro (IL 3.1):** aplica estrategias de ajuste de hiperparámetros para maximizar
el rendimiento y la eficiencia de los modelos seleccionados.

---

### Dónde estamos

El RA2 dejó un modelo que funciona: F1-macro de **0,70**, con un recall de 0,40 en la clase
minoritaria. La pregunta del RA3 es si se puede hacer mejor, y **cómo saber si de verdad
mejoró**.

Hoy: los hiperparámetros. Los parámetros que el modelo **no** aprende de los datos y que hay
que elegir desde fuera: cuántos árboles, qué profundidad, cuántas muestras por hoja.

---

### La idea central, y probablemente te va a decepcionar

> **El ajuste de hiperparámetros da mejoras de segundo orden.**

Las mejoras de primer orden vienen de otra parte: de las variables que elegiste, de cómo
partiste los datos, de haber definido bien el problema. Si el modelo va mal, ajustar
hiperparámetros casi nunca lo salva.

Al final de la sesión vas a haber probado 12 configuraciones distintas y vas a comparar la
mejor contra los valores por defecto. **Guarda tu expectativa** de cuánto vas a ganar.

---

### Dónde se ajusta, que es lo que de verdad se evalúa

Ajustar exige comparar configuraciones, y comparar exige medir. ¿Medir dónde?

- **En entrenamiento:** no sirve. Más complejidad siempre puntúa mejor ahí.
- **En prueba:** es hacer trampa. Si eliges la configuración que mejor puntúa en la prueba, esa
  puntuación deja de estimar el futuro.
- **En validación cruzada dentro del entrenamiento:** correcto. Y aquí, por lo mismo del RA2,
  los pliegues tienen que respetar el segmento.

---

### Al final de la sesión debes entregar

El informe de ajuste: qué espacio exploraste, con qué esquema de validación, **cuánto ganaste**
y si esa ganancia supera la variabilidad entre pliegues.

> ### 🎓 Pauta docente — Actividad 3.1
>
> **6 horas pedagógicas.** La distribución de abajo cubre ~3 h de trabajo guiado; el resto es
> para aplicar el mismo esquema al caso oficial del equipo, que va a la Parcial 3.
>
> | Bloque | Min | Foco |
> |---|---|---|
> | 0 · Encuadre | 15 | Qué es un hiperparámetro y qué no |
> | 1 · Validación cruzada por grupo ⭐ | 40 | Dónde se mide, y por qué GroupKFold |
> | 2 · La búsqueda | 40 | Rejilla vs aleatoria |
> | 3 · ¿Cuánto ganamos? ⭐⭐ | 40 | **El macro se mueve; `LEVEL_2` no** |
> | 4 · La trampa de ajustar en prueba ⭐ | 35 | Más sutil que la fuga del RA2 |
> | Cierre | 10 | Informe de ajuste |
>
> **El bloque 3 es el que sostiene la sesión y no se recorta.** La búsqueda sube el F1-macro
> **+0,0789** (0,5104 → 0,5893) y **sí** supera el ruido (0,0424). No lo adelantes. El remate
> es que `LEVEL_2` sigue en **0,0893**.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

import waymo
RUTA_DATOS = waymo.exigir_detecciones_reales(RAIZ)
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"
print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as supervisado
from kedro_mly1101.pipelines.optimizacion import nodes as optimizacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG, FUGA, AJUSTE = PARAMETROS["modelo"], PARAMETROS["fuga"], PARAMETROS["ajuste"]

# La misma cadena de siempre: limpieza del RA1 -> partición del RA2.
crudo = pd.read_parquet(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

marcada = supervisado.particionar(
    supervisado.preparar_variables(limpio, CONFIG, FUGA), CONFIG
)
entrena = marcada[marcada["particion"] == "entrenamiento"]

print(f"Entrenamiento: {len(entrena):,} filas en {entrena[CONFIG['grupo']].nunique()} segmentos")
print(f"Métrica de trabajo: {AJUSTE['metrica']}  ·  pliegues: {AJUSTE['n_pliegues']}")

---
# Bloque 1 · ⭐ Dónde se mide: validación cruzada por grupo

Para comparar configuraciones hace falta una estimación de desempeño **que no use la prueba**.
Se saca partiendo el entrenamiento en `k` pliegues: se entrena con `k−1` y se mide en el que
queda, `k` veces.

**Y los pliegues tienen que respetar el segmento**, exactamente por lo mismo que la partición
de la Actividad 2.2: las detecciones de un segmento comparten contexto. `GroupKFold` lo
garantiza.

### ✏️ TODO 1 — Comprobar que los pliegues no rompen segmentos

In [ ]:
from sklearn.model_selection import GroupKFold

X = entrena[CONFIG["variables"]]
y = entrena[CONFIG["objetivo"]]
grupos = entrena[CONFIG["grupo"]]

cv = GroupKFold(n_splits=AJUSTE["n_pliegues"])

filas = []
for numero, (idx_entrena, idx_valida) in enumerate(cv.split(X, y, groups=grupos), start=1):
    seg_entrena = set(grupos.iloc[idx_entrena])
    seg_valida = set(grupos.iloc[idx_valida])
    filas.append(
        {
            "pliegue": numero,
            "filas_entrena": len(idx_entrena),
            "filas_valida": len(idx_valida),
            "segmentos_valida": len(seg_valida),
            "segmentos_compartidos": len(seg_entrena & seg_valida),
        }
    )
pd.DataFrame(filas)

In [ ]:
# Autochequeo
tabla = pd.DataFrame(filas)
assert (tabla["segmentos_compartidos"] == 0).all(), (
    "revisa: ningún pliegue puede compartir segmentos. ¿Pasaste groups= al split?"
)
print(f"✅ {len(tabla)} pliegues, 0 segmentos compartidos en todos.")
print("   La estimación que salga de aquí es honesta: cada pliegue evalúa")
print("   sobre segmentos que el modelo nunca vio.")

### ✏️ TODO 2 — La métrica del ajuste

Antes de buscar hay que decidir **qué se está maximizando**. Mira el parámetro
`AJUSTE["metrica"]` y responde.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

¿Por qué se optimiza `f1_macro` y no `accuracy`? *(Pista: revisa lo que descubriste en el
bloque 3 de la Actividad 2.2.)*

> ### 🎓 Pauta docente — Bloque 1
>
> **Cifras medidas:** 5 pliegues sobre 384.280 filas de entrenamiento en **30** segmentos,
> **0 segmentos compartidos** en todos.
>
> **Respuesta al TODO 2:** porque el problema está desbalanceado ~88/12 (en prueba, 81,7 %
> `LEVEL_1`) y la exactitud del baseline trivial ya es 0,8172. Optimizar exactitud llevaría al
> buscador a configuraciones que **abandonan la clase minoritaria**, que es justo la que
> interesa. `f1_macro` pondera las dos clases por igual.
>
> **La frase:** *elegir la métrica del ajuste es elegir qué error te importa. Se decide antes de
> buscar, no después de ver los resultados.*
>
> **Criterio de logro:** verifica los 0 segmentos compartidos y justifica la métrica en términos
> del desbalance.

---
# Bloque 2 · Buscar: rejilla contra búsqueda aleatoria

| | Rejilla (`GridSearchCV`) | Aleatoria (`RandomizedSearchCV`) |
|---|---|---|
| Qué prueba | **Todas** las combinaciones | `n_iter` combinaciones al azar |
| Costo | Producto de las opciones: explota | El que tú decidas |
| Ventaja | Exhaustiva en su rejilla | Con el mismo presupuesto explora más regiones |

Con 4 × 6 × 4 × 3 = **288 combinaciones**, cada una con 5 pliegues, la rejilla exigiría 1.440
entrenamientos. La búsqueda aleatoria hace 12 × 5 = 60.

> **Por qué la aleatoria suele bastar:** casi siempre solo un par de hiperparámetros importan de
> verdad. La rejilla gasta la mayor parte del presupuesto variando los que dan igual.

### ✏️ TODO 3 — Lanzar la búsqueda

In [ ]:
busqueda = optimizacion.buscar_hiperparametros(marcada, CONFIG, AJUSTE)
busqueda.head(6)

### ✏️ TODO 4 — Leer la tabla con desconfianza

Mira las columnas `mean_test_score` y `std_test_score`.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

Compara la diferencia entre la primera y la segunda configuración con la desviación típica
entre pliegues de la primera. ¿Puedes afirmar que la primera es mejor?

> ### 🎓 Pauta docente — Bloque 2
>
> **Cifras medidas** (12 combinaciones, 5 pliegues, `f1_macro`):
>
> | Rango | F1-macro | Desv. | `n_estimators` | `max_depth` | `min_samples_leaf` | `max_features` |
> |---|---|---|---|---|---|---|
> | 1 | **0,6964** | 0,0091 | 400 | 16 | 1 | sqrt |
> | 2 | 0,6933 | 0,0076 | 50 | sin límite | 1 | log2 |
> | 3 | 0,6922 | 0,0078 | 100 | 4 | 20 | sqrt |
> | 4 | 0,6917 | 0,0085 | 100 | 12 | 5 | 0,5 |
>
> **Respuesta al TODO 4:** la diferencia entre la 1.ª y la 2.ª es **0,0031**; la desviación entre
> pliegues de la 1.ª es **0,0091**, casi tres veces mayor. **No hay evidencia de que la primera
> sea mejor.** El "ranking" ordena ruido.
>
> Es la primera vez en el curso que se cuestiona un ranking, y conviene decirlo así:
>
> > *Que scikit-learn te devuelva las configuraciones ordenadas no significa que ese orden
> > signifique algo. Ordenar siempre se puede; distinguir, no siempre.*
>
> **Fíjate además en el desorden del ranking:** 50 árboles sin límite de profundidad queda
> segundo, y 100 árboles con profundidad 4 queda tercero. No hay un patrón claro, que es
> exactamente lo que se espera cuando las diferencias son ruido.
>
> **Criterio de logro:** compara la diferencia contra la desviación y concluye que no son
> distinguibles.

---
# Bloque 3 · ⭐⭐ ¿Cuánto ganamos de verdad?

Ya tenemos la mejor configuración de 12. Ahora la comparación que importa: **contra no haber
ajustado nada**.

### ✏️ TODO 5 — Antes de ejecutar, apuesta

**Creo que el ajuste mejorará el F1-macro en:** `____`

*(Escríbelo. Otra vez.)*

In [ ]:
ganancia = optimizacion.comparar_ajuste_contra_defecto(marcada, CONFIG, AJUSTE, busqueda)
ganancia

In [ ]:
# Autochequeo
delta = ganancia.loc[1, "ganancia"]
ruido = ganancia.loc[0, "desv_entre_pliegues"]
print(f"Ganancia del ajuste : {delta:+.4f}")
print(f"Ruido entre pliegues: {ruido:.4f}")
print()
assert delta > ruido, (
    "revisa: en v2 la ganancia del ajuste debería superar el ruido entre pliegues"
)
print("✅ La ganancia del ajuste es MAYOR que la variabilidad entre pliegues.")
print("   Traducido: 12 configuraciones y el macro sí se mueve.")
print("   El remate está en LEVEL_2: el F1 de las difíciles sigue en 0,0893.")

### ✏️ TODO 6 — Entonces, ¿el ajuste no sirve?

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Fue inútil esta sesión?
2. Si el ajuste da tan poco, ¿de dónde vienen las mejoras grandes en un proyecto de ML?
3. ¿En qué situación **sí** esperarías que el ajuste diera una mejora importante?

> ### 🎓 Pauta docente — Bloque 3 ⭐⭐
>
> **Cifras medidas** (Perception v2, 2026-09-08, `ganancia_del_ajuste.csv`):
>
> | Configuración | F1-macro |
> |---|---|
> | Valores por defecto | **0,5104** |
> | Búsqueda | **0,5893** |
> | **Ganancia** | **+0,0789** (supera ruido 0,0424) |
>
> **El ajuste sí ganó, y aun así el modelo se pierde el 94 % de las difíciles.** Deja que
> celebren el +0,08 y después pregunta por `LEVEL_2`.
>
> **Respuestas esperadas al TODO 6:**
>
> 1. **No fue inútil: el macro subió y superó el ruido.** Reportarlo es correcto. Venderlo como
>    "ya podemos confiar en el sensor" no lo es.
> 2. **`LEVEL_2` sigue en F1 0,0893.** Las mejoras de primer orden siguen en las variables, en
>    más datos de la clase difícil y en el umbral de decisión, no en otra vuelta de la rejilla.
> 3. El ensamble de 3.2 (0,5938) no se distingue del gradient boosting (0,594).
>
> **La frase de la sesión:**
>
> > *Ajustar hiperparámetros es lo último que hay que hacer, y lo primero que todos quieren
> > hacer, porque es lo único que se puede automatizar.*
>
> **Criterio de logro:** reconoce que la ganancia **sí** supera el ruido, no vende eso como
> "el sensor ya es confiable", y ubica las mejoras de primer orden fuera del ajuste (`LEVEL_2`).

---
# Bloque 4 · ⭐ La trampa: ajustar mirando la prueba

Nadie *entrena* con la prueba. Pero mucha gente **elige** mirándola: prueba varias
configuraciones, ve cuál puntúa mejor en el conjunto de prueba y reporta ese número.

El modelo nunca vio esos datos. ¿Cuál es el problema?

Que la puntuación que reportas ya no es una estimación del desempeño futuro: es **el máximo de
una muestra**, y el máximo de una muestra siempre es optimista.

### ✏️ TODO 7 — Medirlo

In [ ]:
fuga = optimizacion.medir_fuga_por_ajustar_en_prueba(marcada, CONFIG, AJUSTE)
fuga

### ✏️ TODO 8 — Tres cosas distintas en una tabla

La tabla mide tres cosas que se confunden con facilidad. Explica cada una.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. **Optimismo:** salió `____`. ¿Significa que la trampa es inofensiva?
2. **Margen de la trampa:** `____`. ¿Qué representa?
3. **Brecha validación vs prueba:** la validación cruzada puntúa sistemáticamente **más bajo**
   que la prueba. ¿Es esto una fuga? ¿Por qué pasa?

> ### 🎓 Pauta docente — Bloque 4 ⭐
>
> **Cifras de la trampa:** no recites una tabla de profundidades del hilo viejo. Salen de la
> celda. Lo que se busca: **no elegir hiperparámetros mirando la prueba.** En v2, el F1-macro
> de referencia del RF por defecto en prueba es **0,4822**; el de la búsqueda, **0,5893**.
>
> **El optimismo salió CERO, y hay que explicarlo bien o se saca la conclusión contraria.**
> Ambos criterios eligieron la misma configuración (*sin límite*), así que esta vez la trampa no
> pagó. **Eso no la vuelve inofensiva: significa que la suerte no la premió.**
>
> **Respuestas al TODO 8:**
>
> 1. No. Un cero aquí es una coincidencia de esta corrida, no una propiedad del método. Es el
>    mismo razonamiento del bloque 2 de la Actividad 2.2: un riesgo que no se manifestó sigue
>    siendo un riesgo.
> 2. **0,0135** es lo que separa la mejor configuración de la peor *en la prueba*: el tamaño del
>    premio por elegir mal. Es más del doble de la ganancia que se buscaba con todo el ajuste
>    (0,0006). Dicho de otro modo: **haciendo trampa se gana veinte veces más que ajustando
>    bien**, y por eso la tentación es real.
> 3. **No es fuga.** La validación cruzada entrena con 4/5 de los datos, así que estima el
>    desempeño de un modelo entrenado con **menos** datos del que finalmente entregas. Es un
>    sesgo **conservador y conocido**, y por eso es un criterio de selección seguro: se equivoca
>    hacia abajo.
>
> **La distinción que hay que dejar clara:** *"me da un número más bajo"* y *"me engaña"* no son
> lo mismo. La validación cruzada hace lo primero; ajustar en prueba hace lo segundo.
>
> **Criterio de logro:** distingue las tres magnitudes, explica por qué un optimismo de cero no
> exonera al método, e identifica la brecha validación/prueba como sesgo conservador y no fuga.

---
# Cierre · Informe de ajuste

**Modelo ajustado:** `____` · **Métrica optimizada:** `____` · **Por qué esa métrica:** `____`

### Esquema de validación

| Campo | Valor |
|---|---|
| Tipo de validación | `____` |
| Pliegues | `____` |
| Variable de agrupación | `____` |
| Segmentos compartidos entre pliegues | `____` |

### Espacio explorado

| Hiperparámetro | Valores | Por qué ese rango |
|---|---|---|
| `____` | | |
| `____` | | |

**Estrategia** (rejilla o aleatoria) **y por qué:** `____`
**Combinaciones probadas:** `____` de `____` posibles

### El resultado

| | F1-macro | Desv. entre pliegues |
|---|---|---|
| Valores por defecto | `____` | `____` |
| Mejor configuración | `____` | `____` |
| **Ganancia** | `____` | |

**¿La ganancia supera la variabilidad entre pliegues?** `____`
**Conclusión:** `____`

> Si tu ganancia no supera el ruido, **dilo**. Reportar una mejora que no puedes distinguir del
> azar es el error que esta sesión existe para evitar.

### Dónde buscaría la próxima mejora

`____`

*(Y por qué ahí y no en más ajuste.)*

> ### 🎓 Criterios de logro — Actividad 3.1 (IL 3.1)
>
> | Nivel | Descripción |
> |---|---|
> | **Destacado (4)** | Todo lo del 3, y además: **contrasta la ganancia contra el ruido** y aun así no vende `LEVEL_2` como resuelto; explica por qué un optimismo de cero no exonera la trampa; distingue la brecha validación/prueba de una fuga |
> | **Logrado (3)** | Valida con `GroupKFold` y 0 segmentos compartidos; justifica la métrica por el desbalance; compara contra los valores por defecto y contra la desviación entre pliegues |
> | **En desarrollo (2)** | Ejecuta la búsqueda y reporta la mejor configuración, pero presenta la ganancia como mejora sin contrastarla con el ruido |
> | **Inicial (1)** | Ajusta sobre la prueba, o valida sin agrupar por segmento |
>
> **Lo primero que hay que mirar al corregir:** si el informe dice *"el ajuste resolvió el
> modelo"*. El macro subió y superó el ruido; `LEVEL_2` sigue en 0,0893. Ese es el error que la
> sesión previene: vender la métrica global.